# Practical Assignment: Support Vector Machines (SVM), PCA, and LDA (40 Points)

## Grading Distribution & Assessment Criteria

| Section | Target Task | Evaluation Metric | Points |
| :--- | :--- | :--- | :---: |
| **Part 1.1** | `solve_hard_margin_q1` (Hard-Margin SVM Numerical Verification) | Exact match for $w$, $w_0$, margin width, and test prediction | 4 |
| **Part 1.2** | `run_soft_margin_experiment` (Penalty $C$ Trade-off) | Margin width monotonicity & support vector count assertions | 6 |
| **Part 2.1** | `explicit_poly_map` & `poly_kernel_matrix` | Gram matrix equivalence ($\|\Phi(U)\Phi(V)^T - \kappa(U,V)\| < 10^{-6}$) | 4 |
| **Part 2.2** | `evaluate_rbf_gammas` (RBF $\gamma$ Overfitting Analysis) | Training memorization & generalization gap validation | 6 |
| **Part 3.1** | `PCAFromScratch` (`fit` & `transform`) | Eigenvalues, variance ratio, and PC1 projection match | 5 |
| **Part 3.2** | `compare_pca_scaling` (Raw vs. Standardized Wine Data) | Cumulative variance threshold ($d \ge 85\%$) dimensionality check | 5 |
| **Part 4.1** | `compare_pca_lda_q5` (PCA Failure vs. Regularized LDA) | Within-class collapse & class-mean separation assertions | 4 |
| **Part 4.2** | `iris_pca_vs_lda_experiment` (Rank Constraint & Iris 1D/2D) | Rank constraint ($K - 1$) & 1D classifier accuracy comparison | 6 |
| **Total** | | | **40** |

## Objectives
1. Empirically verify maximal margin geometry, support vectors, and the bias-variance trade-off controlled by hyperparameter $C$ in Soft-Margin SVM.
2. Implement polynomial feature mappings and kernel functions from scratch, and analyze overfitting dynamics of the RBF kernel parameter $\gamma$.
3. Implement Principal Component Analysis (PCA) from scratch using NumPy and investigate feature scaling sensitivity and cumulative explained variance.
4. Compare unsupervised (PCA) and supervised (LDA) dimensionality reduction in terms of class separability and rank constraints.

## Submission & Execution Rules
- This notebook is fully compatible with **Google Colab** and **Kaggle** (CPU runtime is sufficient).
- Complete all sections marked with `### YOUR CODE HERE`.
- Do not modify or delete the verification (`assert`) cells.
- Before submitting, run `Restart Session and Run All` to ensure sequential execution without errors.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Reproducibility configuration
np.random.seed(42)
plt.rcParams['figure.dpi'] = 100
print("Environment configured successfully. Ready for execution on Colab/Kaggle.")

## Part 1: SVM Geometry, Support Vectors, and Hyperparameter $C$ (10 Points)

### 1.1 Numerical Verification of Hard-Margin SVM (4 Points)
In Theoretical Question 1, we analyzed a 2D linearly separable dataset:
- Positive class ($y = +1$): $x^{(1)} = (2, 2)^T, \; x^{(2)} = (3, 3)^T$
- Negative class ($y = -1$): $x^{(3)} = (0, 0)^T, \; x^{(4)} = (-1, 1)^T$

Fit a linear SVM with a very large penalty parameter (`C=1e6` to approximate Hard-Margin SVM) and compute:
1. Optimal weight vector $w = (w_1, w_2)^T$ and bias $w_0$.
2. Total margin width $\frac{2}{\|w\|_2}$.
3. Predicted label for $x_{\text{test}} = (2, 1)^T$.

In [ ]:
X_q1 = np.array([
    [2.0, 2.0],
    [3.0, 3.0],
    [0.0, 0.0],
    [-1.0, 1.0]
])
y_q1 = np.array([1, 1, -1, -1])
x_test_q1 = np.array([[2.0, 1.0]])

def solve_hard_margin_q1(X, y, x_test):
    """
    Fits a linear SVM with C=1e6 and returns:
    w (shape (2,)), w0 (float), margin_width (float), test_pred (int)
    """
    ### YOUR CODE HERE
    raise NotImplementedError("Implement this section.")
    ### END YOUR CODE
    return w, w0, margin_width, test_pred

w_q1, w0_q1, margin_q1, pred_q1 = solve_hard_margin_q1(X_q1, y_q1, x_test_q1)

# Assertions matching Theoretical Question 1 analytical results
np.testing.assert_allclose(w_q1, np.array([0.5, 0.5]), atol=1e-3, err_msg="Weight vector w mismatch")
np.testing.assert_allclose(w0_q1, -1.0, atol=1e-3, err_msg="Bias w0 mismatch")
np.testing.assert_allclose(margin_q1, 2.0 * np.sqrt(2.0), atol=1e-2, err_msg="Margin width mismatch")
assert pred_q1 == 1, f"Expected prediction +1 for (2,1), got {pred_q1}"

print(f"Part 1.1 Passed | w = {np.round(w_q1, 3)}, w0 = {w0_q1:.3f}, Margin = {margin_q1:.3f}, Test Pred = {pred_q1}")

### 1.2 Controlled Experiment: Effect of Penalty Parameter $C$ in Soft-Margin SVM (6 Points)
Referring to Slide 43 of `4-NM-ML-SVM` and Theoretical Question 2(c):
- A **small $C$** tolerates margin violations ($\xi_n > 0$), producing a **wider margin**, more support vectors, higher bias, and lower variance.
- A **large $C$** heavily penalizes violations, producing a **narrower margin**, fewer support vectors, lower bias, and higher variance.

Implement `run_soft_margin_experiment(X, y, C_list)` to train linear SVMs across $C \in \{0.01, 1.0, 100.0\}$, record the margin widths and support vector counts, and visualize the decision boundaries and margin hyperplanes ($w^T x + w_0 = \pm 1$).

In [ ]:
# Generate overlapping 2D dataset
X_blobs, y_blobs = datasets.make_blobs(
    n_samples=120, centers=[[-1.2, -1.2], [1.2, 1.2]],
    cluster_std=1.1, random_state=42
)
y_blobs = np.where(y_blobs == 0, -1, 1)

def run_soft_margin_experiment(X, y, C_list):
    """
    Trains linear SVMs for each C in C_list.
    Returns:
      models: dict {C: fitted SVC}
      margins: dict {C: margin_width}
      sv_counts: dict {C: number of support vectors}
    """
    models, margins, sv_counts = {}, {}, {}
    ### YOUR CODE HERE
    raise NotImplementedError("Implement this section.")
    ### END YOUR CODE
    return models, margins, sv_counts

C_values = [0.01, 1.0, 100.0]
models_c, margins_c, sv_counts_c = run_soft_margin_experiment(X_blobs, y_blobs, C_values)

# Visualization of decision boundaries and margins
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
x_min, x_max = X_blobs[:, 0].min() - 0.5, X_blobs[:, 0].max() + 0.5
y_min, y_max = X_blobs[:, 1].min() - 0.5, X_blobs[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))

for ax, c_val in zip(axes, C_values):
    clf = models_c[c_val]
    Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z > 0, alpha=0.2, cmap=plt.cm.coolwarm)
    ax.contour(xx, yy, Z, colors='k', levels=[-1, 0, 1], linestyles=['--', '-', '--'])
    ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap=plt.cm.coolwarm, edgecolors='k', s=30)
    ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
               s=110, facecolors='none', edgecolors='limegreen', linewidths=1.5, label='Support Vectors')
    ax.set_title(f"C = {c_val}\nMargin = {margins_c[c_val]:.2f} | SVs = {sv_counts_c[c_val]}")
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

# Assertions verifying theoretical relationship between C, margin width, and support vectors
assert margins_c[0.01] > margins_c[1.0] > margins_c[100.0], "Margin must strictly decrease as C increases"
assert sv_counts_c[0.01] > sv_counts_c[100.0], "Smaller C must yield more support vectors due to wider margin"
print("Part 1.2 Passed | Margins:", {k: round(v, 3) for k, v in margins_c.items()}, "| SV counts:", sv_counts_c)

> ### Expected Output Reference: Soft-Margin SVM & Penalty $C$
> Verify your generated plots against the reference output below:
- **Small $C$ (0.01):** Wide margin with many support vectors (high bias, low variance).
- **Large $C$ (100.0):** Narrow margin tightly fitted to boundary points (low bias, high variance).
>
> <p align="center">
>   <img src="https://raw.githubusercontent.com/rallm/iust-ml-mozayani-fall2026/main/HW3-SVM-PCA-LDA/assets/svm_c_comparison_ref.png" width="850"/>
> </p>

## Part 2: Kernel Trick and RBF Kernel $\gamma$ Overfitting Analysis (10 Points)

### 2.1 Explicit Feature Mapping vs. Polynomial Kernel Function (4 Points)
In Theoretical Question 3(b), we proved that for $x = (x_1, x_2)^T \in \mathbb{R}^2$, the degree-2 polynomial kernel:
$$\kappa(u, v) = (u^T v + 1)^2$$
is identical to the inner product $\phi(u)^T \phi(v)$ under the 6-dimensional explicit mapping:
$$\phi(x) = \left(1, \sqrt{2}x_1, \sqrt{2}x_2, x_1^2, x_2^2, \sqrt{2}x_1 x_2\right)^T$$

Implement both `explicit_poly_map(X)` and `poly_kernel_matrix(U, V)` using pure NumPy and verify that $\Phi(U)\Phi(V)^T = \kappa(U, V)$.

In [ ]:
def explicit_poly_map(X):
    """
    Maps X of shape (N, 2) to Phi(X) of shape (N, 6):
    [1, sqrt(2)*x1, sqrt(2)*x2, x1^2, x2^2, sqrt(2)*x1*x2]
    """
    ### YOUR CODE HERE
    raise NotImplementedError("Implement this section.")
    ### END YOUR CODE
    return Phi

def poly_kernel_matrix(U, V):
    """
    Computes Gram matrix K of shape (N_u, N_v) where K[i, j] = (u_i^T v_j + 1)^2
    without explicitly constructing Phi.
    """
    ### YOUR CODE HERE
    raise NotImplementedError("Implement this section.")
    ### END YOUR CODE
    return K

# Verify with Question 3(b) vectors and random batch matrices
u_q3 = np.array([[1.0, 2.0]])
v_q3 = np.array([[3.0, -1.0]])
assert np.isclose(poly_kernel_matrix(u_q3, v_q3)[0, 0], 4.0), "Kernel value for Question 3(b) must be 4.0"

U_rand = np.random.randn(15, 2)
V_rand = np.random.randn(20, 2)
K_explicit = np.dot(explicit_poly_map(U_rand), explicit_poly_map(V_rand).T)
K_trick = poly_kernel_matrix(U_rand, V_rand)

np.testing.assert_allclose(K_explicit, K_trick, atol=1e-6, err_msg="Explicit dot product and kernel matrix must match")
print("Part 2.1 Passed | Kernel trick equivalence verified with max diff:", np.max(np.abs(K_explicit - K_trick)))

### 2.2 Controlled Experiment: RBF Kernel Parameter $\gamma$ and Overfitting (6 Points)
The Gaussian RBF kernel is defined as:
$$\kappa(x^{(i)}, x^{(j)}) = \exp\left(-\gamma \|x^{(i)} - x^{(j)}\|^2\right), \quad \gamma = \frac{1}{2\sigma^2}$$
Referring to Slide 55 of `4-NM-ML-SVM` and Theoretical Question 3(d):
- Small $\gamma$ corresponds to large $\sigma$ (smooth, global decision boundary; risk of underfitting).
- Very large $\gamma$ corresponds to tiny $\sigma$ (tight local islands around training points; high variance / overfitting).

Train `SVC(kernel='rbf', C=1.0, gamma=g)` for $\gamma \in \{0.1, 1.0, 50.0\}$ on a noisy non-linear `make_moons` dataset, plot the decision boundaries, and compute training vs. validation accuracies.

In [ ]:
X_moons, y_moons = datasets.make_moons(n_samples=240, noise=0.28, random_state=42)
X_m_train, X_m_val, y_m_train, y_m_val = train_test_split(
    X_moons, y_moons, test_size=0.3, random_state=42, stratify=y_moons
)

def evaluate_rbf_gammas(X_tr, y_tr, X_va, y_va, gamma_list):
    """
    Trains RBF SVMs for each gamma in gamma_list.
    Returns:
      models: dict {gamma: fitted SVC}
      train_accs: dict {gamma: float}
      val_accs: dict {gamma: float}
    """
    models, train_accs, val_accs = {}, {}, {}
    ### YOUR CODE HERE
    raise NotImplementedError("Implement this section.")
    ### END YOUR CODE
    return models, train_accs, val_accs

gamma_values = [0.1, 1.0, 50.0]
models_g, train_accs_g, val_accs_g = evaluate_rbf_gammas(X_m_train, y_m_train, X_m_val, y_m_val, gamma_values)

# Plot decision boundaries
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))

for ax, g in zip(axes, gamma_values):
    clf = models_g[g]
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap=plt.cm.coolwarm)
    ax.scatter(X_m_train[:, 0], X_m_train[:, 1], c=y_m_train, cmap=plt.cm.coolwarm, edgecolors='k', s=25, label='Train')
    ax.scatter(X_m_val[:, 0], X_m_val[:, 1], c=y_m_val, cmap=plt.cm.coolwarm, marker='^', edgecolors='k', s=35, label='Val')
    ax.set_title(f"RBF gamma = {g}\nTrain Acc: {train_accs_g[g]:.3f} | Val Acc: {val_accs_g[g]:.3f}")
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")

plt.tight_layout()
plt.show()

gap_optimal = train_accs_g[1.0] - val_accs_g[1.0]
gap_overfit = train_accs_g[50.0] - val_accs_g[50.0]
assert train_accs_g[50.0] > 0.95, "Very large gamma must memorize training samples"
assert gap_overfit > gap_optimal, "Very large gamma (50.0) must exhibit a larger generalization gap than gamma=1.0"
print(f"Part 2.2 Passed | Optimal Gap (gamma=1.0): {gap_optimal:.3f}, Overfit Gap (gamma=50.0): {gap_overfit:.3f}")

> ### Expected Output Reference: RBF Kernel $\gamma$ Dynamics
> Verify your generated plots against the reference output below:
- **$\gamma = 0.1$:** Smooth, near-linear boundary (underfitting).
- **$\gamma = 1.0$:** Balanced non-linear separation following the two moons.
- **$\gamma = 50.0$:** Tight islands around individual training points (severe overfitting).
>
> <p align="center">
>   <img src="https://raw.githubusercontent.com/rallm/iust-ml-mozayani-fall2026/main/HW3-SVM-PCA-LDA/assets/rbf_gamma_comparison_ref.png" width="850"/>
> </p>

## Part 3: Principal Component Analysis (PCA) from Scratch & Scaling Sensitivity (10 Points)

### 3.1 Pure NumPy Implementation of PCA (5 Points)
Following Slide 14 of `9-NM-ML-PCAnLDA`:
1. Compute the mean vector $\mu = \frac{1}{n}\sum_{i=1}^{n} x_i$ and center the data: $\tilde{X} = X - \mu$.
2. Compute the MLE covariance matrix $C = \frac{1}{n}\sum_{i=1}^{n}(x_i - \mu)(x_i - \mu)^T$.
3. Compute eigenvalues and eigenvectors of $C$ (`np.linalg.eigh`) and sort them in descending order $\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_m$.
4. Project centered data onto the top $d$ eigenvectors: $Y = \tilde{X} U_d$.
5. Compute the explained variance ratio $\frac{\lambda_j}{\sum_{k=1}^{m}\lambda_k}$.

Implement `PCAFromScratch` below and verify it against the exact covariance matrix from Theoretical Question 4.

In [ ]:
class PCAFromScratch:
    def __init__(self):
        self.mean_ = None
        self.eigenvalues_ = None
        self.components_ = None  # Columns are unit eigenvectors sorted by descending eigenvalue
        self.explained_variance_ratio_ = None

    def fit(self, X):
        ### YOUR CODE HERE
        raise NotImplementedError("Implement this section.")
        ### END YOUR CODE
        return self

    def transform(self, X, n_components):
        ### YOUR CODE HERE
        raise NotImplementedError("Implement this section.")
        ### END YOUR CODE

# Construct synthetic centered dataset with exact covariance matrix C = [[5, 2], [2, 2]] from Question 4
C_q4 = np.array([[5.0, 2.0], [2.0, 2.0]])
L_chol = np.linalg.cholesky(C_q4)
Z_base = np.random.randn(5000, 2)
Z_base = (Z_base - Z_base.mean(axis=0)) @ np.linalg.inv(np.linalg.cholesky(np.cov(Z_base, rowvar=False, bias=True))).T
X_q4_sim = Z_base @ L_chol.T

pca_q4 = PCAFromScratch().fit(X_q4_sim)

# Verify eigenvalues, explained variance ratio, and projection of x = [3, -1]
np.testing.assert_allclose(pca_q4.eigenvalues_, np.array([6.0, 1.0]), atol=1e-2, err_msg="Eigenvalues mismatch")
np.testing.assert_allclose(pca_q4.explained_variance_ratio_, np.array([6.0/7.0, 1.0/7.0]), atol=1e-2)

proj_sample = pca_q4.transform(np.array([[3.0, -1.0]]), n_components=1)
np.testing.assert_allclose(np.abs(proj_sample[0, 0]), np.sqrt(5.0), atol=1e-2, err_msg="Projection onto PC1 mismatch")
print(f"Part 3.1 Passed | Eigenvalues: {np.round(pca_q4.eigenvalues_, 3)}, |Proj([3,-1])|: {np.abs(proj_sample[0,0]):.3f}")

### 3.2 Controlled Experiment: Feature Scaling Sensitivity on the Wine Dataset (5 Points)
Referring to Slide 10 and Slide 21 of `9-NM-ML-PCAnLDA` and Theoretical Question 4(c)–(d):
- PCA maximizes variance; if features have vastly different numerical units (e.g., `Proline` $\approx 1000$ vs. `Hue` $\approx 1$ in the Wine dataset), unscaled PCA will align PC1 almost entirely with the largest-scale feature.
- Standardizing features via $x_{\text{stand}} = \frac{x - \mu}{\sigma}$ eliminates unit bias.

Apply `PCAFromScratch` to both **raw (unstandardized)** and **standardized** `Wine` features, compute how many components ($d$) are needed to retain at least **85% cumulative variance** (Slide 14), and plot the Scree / Cumulative Variance curves.

In [ ]:
wine = datasets.load_wine()
X_wine = wine.data

def compare_pca_scaling(X, variance_threshold=0.85):
    """
    Fits PCAFromScratch on raw X and standardized X.
    Returns:
      pca_raw: fitted PCAFromScratch on raw X
      pca_scaled: fitted PCAFromScratch on standardized X
      d_raw: int, minimum components to reach variance_threshold on raw X
      d_scaled: int, minimum components to reach variance_threshold on scaled X
    """
    ### YOUR CODE HERE
    raise NotImplementedError("Implement this section.")
    ### END YOUR CODE
    return pca_raw, pca_scaled, d_raw, d_scaled

pca_wine_raw, pca_wine_std, d_wine_raw, d_wine_std = compare_pca_scaling(X_wine, variance_threshold=0.85)

# Scree & Cumulative Variance Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
comps = np.arange(1, X_wine.shape[1] + 1)

axes[0].bar(comps, pca_wine_raw.explained_variance_ratio_ * 100, alpha=0.7, label='Individual')
axes[0].plot(comps, np.cumsum(pca_wine_raw.explained_variance_ratio_) * 100, 'ro-', label='Cumulative')
axes[0].axhline(85, color='k', linestyle='--', label='85% Threshold')
axes[0].set_title(f"Unstandardized Wine Data\nPC1 = {pca_wine_raw.explained_variance_ratio_[0]*100:.1f}% | d(>=85%) = {d_wine_raw}")
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Explained Variance (%)")
axes[0].legend()

axes[1].bar(comps, pca_wine_std.explained_variance_ratio_ * 100, alpha=0.7, color='teal', label='Individual')
axes[1].plot(comps, np.cumsum(pca_wine_std.explained_variance_ratio_) * 100, 'ro-', label='Cumulative')
axes[1].axhline(85, color='k', linestyle='--', label='85% Threshold')
axes[1].set_title(f"Standardized Wine Data\nPC1 = {pca_wine_std.explained_variance_ratio_[0]*100:.1f}% | d(>=85%) = {d_wine_std}")
axes[1].set_xlabel("Principal Component")
axes[1].set_ylabel("Explained Variance (%)")
axes[1].legend()

plt.tight_layout()
plt.show()

assert d_wine_raw == 1, "Without scaling, PC1 alone captures >99% variance due to Proline scale"
assert d_wine_std >= 5, f"After standardization, at least 5 components are needed for 85% variance, got {d_wine_std}"
print(f"Part 3.2 Passed | Components for >=85% variance -> Unscaled: {d_wine_raw}, Standardized: {d_wine_std}")

> ### Expected Output Reference: Feature Scaling Sensitivity in PCA
> Verify your generated plots against the reference output below:
- **Unstandardized (Left):** PC1 dominates (>99% variance) solely due to the large numerical scale of `Proline` ($d = 1$).
- **Standardized (Right):** Variance is distributed across standardized features, requiring $d \ge 5$ components to reach 85%.
>
> <p align="center">
>   <img src="https://raw.githubusercontent.com/rallm/iust-ml-mozayani-fall2026/main/HW3-SVM-PCA-LDA/assets/pca_scaling_ref.png" width="850"/>
> </p>

## Part 4: Unsupervised PCA vs. Supervised LDA (10 Points)

### 4.1 Reproducing Theoretical Question 5: When PCA Fails at Classification (4 Points)
Consider the 4-point dataset from Theoretical Question 5:
- Class $\omega_1$ (`y=0`): $(1, 0)^T, \; (1, 6)^T$
- Class $\omega_2$ (`y=1`): $(3, 0)^T, \; (3, 6)^T$

Notice that the within-class variance along $x_1$ is zero, making $S_W$ singular (as seen in $u_2^T S_W u_2 = 0$ in Question 5(b)). Therefore, in `LinearDiscriminantAnalysis`, we use `solver='eigen'` with covariance regularization (`shrinkage=0.01`) to ensure $S_W$ is invertible. Project these points onto 1D using PCA and LDA, and verify that PCA collapses both classes onto identical coordinates whereas LDA separates them completely.

In [ ]:
X_q5 = np.array([
    [1.0, 0.0],
    [1.0, 6.0],
    [3.0, 0.0],
    [3.0, 6.0]
])
y_q5 = np.array([0, 0, 1, 1])

def compare_pca_lda_q5(X, y):
    """
    Projects X onto 1D using PCA and regularized LDA.
    Returns X_pca_1d (shape (4, 1)) and X_lda_1d (shape (4, 1)).
    """
    ### YOUR CODE HERE
    raise NotImplementedError("Implement this section.")
    ### END YOUR CODE
    return X_pca_1d, X_lda_1d

X_q5_pca, X_q5_lda = compare_pca_lda_q5(X_q5, y_q5)

# Verify PCA overlap (class 0 projections equal class 1 projections) and LDA separation
np.testing.assert_allclose(X_q5_pca[:2], X_q5_pca[2:], atol=1e-6, err_msg="PCA projections of both classes must overlap")
assert np.isclose(X_q5_lda[0, 0], X_q5_lda[1, 0], atol=1e-6), "LDA must collapse within-class variance to 0"
assert np.abs(X_q5_lda[0, 0] - X_q5_lda[2, 0]) > 1.0, "LDA must separate class means"

print("Part 4.1 Passed | PCA 1D:", np.round(X_q5_pca.ravel(), 3), "| LDA 1D:", np.round(X_q5_lda.ravel(), 3))

### 4.2 Controlled Experiment on Iris: Rank Constraint ($K-1$) and Classification Separability (6 Points)
Referring to Slide 30 and Slide 33 of `9-NM-ML-PCAnLDA` and Theoretical Question 5(d):
1. Because $\text{rank}(S_B) \le K - 1$, LDA can extract at most $\min(d, K - 1)$ non-zero discriminant axes. On the Iris dataset ($K = 3$ classes, $d = 4$ features), LDA allows at most $3 - 1 = 2$ components.
2. Project the Iris dataset onto 2D using both PCA and LDA for visual comparison, and evaluate a linear classifier (`LogisticRegression`) trained on the **1D projection** of PCA vs. the **1D projection** of LDA.

In [ ]:
iris = datasets.load_iris()
X_iris, y_iris = iris.data, iris.target

X_ir_train, X_ir_test, y_ir_train, y_ir_test = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)

def iris_pca_vs_lda_experiment(X_tr, y_tr, X_te, y_te):
    """
    1. Standardizes features using StandardScaler (fit on X_tr only).
    2. Fits 2D PCA and 2D LDA on scaled training data for visualization.
    3. Fits 1D PCA and 1D LDA, trains LogisticRegression on each 1D representation,
       and returns test accuracies (acc_pca_1d, acc_lda_1d) along with 2D projections.
    """
    ### YOUR CODE HERE
    raise NotImplementedError("Implement this section.")
    ### END YOUR CODE
    return acc_pca_1d, acc_lda_1d, X_tr_pca2, X_tr_lda2

acc_pca_ir, acc_lda_ir, X_ir_pca2, X_ir_lda2 = iris_pca_vs_lda_experiment(
    X_ir_train, y_ir_train, X_ir_test, y_ir_test
)

# Visualization of 2D PCA vs 2D LDA on Iris
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for cls_idx, cls_name in enumerate(iris.target_names):
    axes[0].scatter(X_ir_pca2[y_ir_train == cls_idx, 0], X_ir_pca2[y_ir_train == cls_idx, 1], label=cls_name, alpha=0.8)
    axes[1].scatter(X_ir_lda2[y_ir_train == cls_idx, 0], X_ir_lda2[y_ir_train == cls_idx, 1], label=cls_name, alpha=0.8)

axes[0].set_title(f"PCA 2D Projection (Iris)\n1D Classifier Test Acc: {acc_pca_ir:.3f}")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].legend()

axes[1].set_title(f"LDA 2D Projection (Iris)\n1D Classifier Test Acc: {acc_lda_ir:.3f}")
axes[1].set_xlabel("LD1")
axes[1].set_ylabel("LD2")
axes[1].legend()

plt.tight_layout()
plt.show()

# Verify K - 1 rank constraint of LDA (K = 3 => n_components = 3 must raise ValueError)
rank_error_caught = False
try:
    LinearDiscriminantAnalysis(n_components=3).fit(X_ir_train, y_ir_train)
except ValueError:
    rank_error_caught = True

assert rank_error_caught, "LDA must raise ValueError when n_components > K - 1"
assert acc_lda_ir > acc_pca_ir, f"1D LDA accuracy ({acc_lda_ir:.3f}) must exceed 1D PCA accuracy ({acc_pca_ir:.3f})"
print(f"Part 4.2 Passed | Rank constraint (K-1=2) verified | 1D PCA Acc: {acc_pca_ir:.3f}, 1D LDA Acc: {acc_lda_ir:.3f}")

> ### Expected Output Reference: Unsupervised PCA vs. Supervised LDA
> Verify your generated plots against the reference output below:
- **PCA (Left):** Maximizes total feature variance without class label awareness.
- **LDA (Right):** Maximizes Fisher's class separability criterion within at most $K - 1$ discriminant axes.
>
> <p align="center">
>   <img src="https://raw.githubusercontent.com/rallm/iust-ml-mozayani-fall2026/main/HW3-SVM-PCA-LDA/assets/pca_vs_lda_ref.png" width="850"/>
> </p>